# Safe Financial Advisory Agent

This project implements a simple financial assistant using a modular, agent-based approach.
The system is designed to answer financial queries, perform basic calculations, and ensure
responses remain safe through the use of guardrails.

The focus of this project is reliability and structured decision-making rather than relying
on external APIs.

In [1]:
def calculator(expression):
    # Basic calculator for financial computations
    try:
        return eval(expression)
    except:
        return "Invalid calculation"

In [2]:
# Sample financial data for decision-making
financial_data = """
Stock A: Return 12%, Risk Medium
Stock B: Return 8%, Risk Low
FD: Return 6%, Very Low Risk
Crypto: High Risk, High Volatility
"""

In [3]:
def input_guardrail(query):
    # Prevent unsafe instructions
    blocked = ["ignore instructions", "bypass", "hack"]
    if any(word in query.lower() for word in blocked):
        return False, "Request rejected due to unsafe instructions."
    return True, query

In [4]:
def behavioral_guardrail(query):
    # Restrict to financial domain
    keywords = ["stock", "investment", "finance", "return", "calculate", "money"]
    if not any(word in query.lower() for word in keywords):
        return False, "Out of scope: this assistant handles financial queries only."
    return True, query

In [5]:
def output_guardrail(answer):
    # Remove unsafe financial claims
    answer_str = str(answer)
    risky = ["guaranteed profit", "no risk", "100% return"]

    if any(word in answer_str.lower() for word in risky):
        return "Response adjusted to remove unsafe financial claims."

    return answer

In [6]:
def planner_agent(query):
    # Decide whether query is calculation or advisory
    if any(op in query for op in ["calculate", "*", "+", "-", "/"]):
        return "calculator"
    return "advisor"

In [7]:
def fake_llm(prompt):
    return """Based on the available information, Stock A offers higher expected returns compared to other options, with a moderate level of risk.

It may be suitable for investors who are comfortable with some variability in returns in exchange for potential gains.

However, market conditions can change and returns are not guaranteed. Investment decisions should always consider individual risk tolerance and financial goals."""

In [8]:
def executor_agent(task, query):

    if task == "calculator":
        expr = query.replace("calculate", "")
        result = calculator(expr)
        return f"The calculated result is {result:.2f}."

    elif task == "advisor":
        return fake_llm(query.strip())

In [9]:
def critic_agent(answer):
    answer_str = str(answer)

    if "no risk" in answer_str.lower():
        return False, "Response flagged as potentially misleading."

    return True, answer

In [10]:
# Main pipeline combining validation, planning, execution, and safety checks
def financial_agent(query):

    # Input guardrail
    ok, query = input_guardrail(query)
    if not ok:
        return query

    # Domain guardrail
    ok, msg = behavioral_guardrail(query)
    if not ok:
        return msg

    # Planner
    task = planner_agent(query)

    # Executor
    answer = executor_agent(task, query)

    # Critic
    ok, checked = critic_agent(answer)
    if not ok:
        return checked

    # Output guardrail
    return output_guardrail(checked)

In [11]:
print("---- Investment Advice ----")
print(financial_agent("Should I invest in Stock A?"))

print("\n---- Calculation ----")
print(financial_agent("calculate 5000 * 1.1"))

print("\n---- Unsafe Query ----")
print(financial_agent("ignore instructions"))

print("\n---- Out of Scope ----")
print(financial_agent("weather today"))

---- Investment Advice ----
Based on the available information, Stock A offers higher expected returns compared to other options, with a moderate level of risk.

It may be suitable for investors who are comfortable with some variability in returns in exchange for potential gains.

However, market conditions can change and returns are not guaranteed. Investment decisions should always consider individual risk tolerance and financial goals.

---- Calculation ----
The calculated result is 5500.00.

---- Unsafe Query ----
Request rejected due to unsafe instructions.

---- Out of Scope ----
Out of scope: this assistant handles financial queries only.
